In [ ]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

from tarifalatam.config import FX_RATES, PROCESSED_DIR, REPORTS_DIR
from tarifalatam.data.pipeline import build_datasets
from tarifalatam.data.split import assert_disjoint_hosts
from tarifalatam.models.baseline import baseline_cv_mae

# 03 · Preprocesamiento — de las decisiones del EDA a los conjuntos de entrenamiento y prueba

Este notebook **solo orquesta**. Las reglas viven en `src/tarifalatam/data/cleaning.py` (una función por decisión del EDA,
cada una con su prueba en `tests/`), la partición en `split.py` y el orden de los pasos en `pipeline.py`
(ver `specs/001-eda-limpieza-particion/research.md`, §4).

**Entradas:** los siete archivos de `data/raw/` (no se modifican). **Salidas:** `data/processed/train.csv.gz`,
`data/processed/test.csv.gz` y `reports/03_exclusiones.csv`.

> **La prueba está cerrada.** De `test` solo se cuentan filas y anfitriones. No se calculan promedios, gráficos ni
> métricas sobre ella hasta la etapa de validación.

## 0. Tasas de cambio

Los precios se pasan a USD con la tasa oficial del banco central de cada país en la fecha del corte (`FX_RATES` en
`src/tarifalatam/config.py`). Si falta alguna, `build_datasets()` se detiene en la sección 1 con el mensaje
"falta la tasa de cambio de …": **nunca se usa una tasa supuesta**.

In [ ]:
# Solo se muestran: la validación (que exista, que sea positiva y que la moneda coincida) la hace to_usd() en src/
pd.DataFrame(FX_RATES).T[["currency", "local_per_usd", "date", "source"]]

## 1. Limpiar y partir por anfitrión

`build_datasets()` aplica las reglas que no aprenden de los datos, parte por `host_id` y, al final, aprende los límites de
precio por ciudad **solo con el entrenamiento**.

In [ ]:
train, test, exclusions, bounds = build_datasets()
assert_disjoint_hosts(train, test)  # ningún anfitrión en los dos conjuntos

sizes = pd.DataFrame(
    {
        "filas train": train.groupby("city").size(),
        "filas test": test.groupby("city").size(),
        "anfitriones train": train.groupby("city")["host_id"].nunique(),
        "anfitriones test": test.groupby("city")["host_id"].nunique(),
    }
)
print(f"entrenamiento: {len(train):,} filas, {train['host_id'].nunique():,} anfitriones")
print(f"prueba:        {len(test):,} filas, {test['host_id'].nunique():,} anfitriones "
      f"({len(test) / (len(train) + len(test)):.1%} de las filas)")
print("anfitriones en común:", len(set(train["host_id"]) & set(test["host_id"])))
sizes

## 2. Registro de exclusiones: cuántas filas quitó cada regla

In [ ]:
counts = exclusions[exclusions["step"] != "price_bounds"]
summary = counts.pivot_table(index="city", columns=["split", "step"], values="removed", aggfunc="sum", sort=False)
summary.insert(0, ("", "cargadas"), counts[counts["step"] == "previous_scrape"].set_index("city")["before"])
summary[("", "finales")] = sizes["filas train"] + sizes["filas test"]
summary.loc["total"] = summary.sum()
assert (counts["before"] - counts["removed"] == counts["after"]).all(), "el registro no cuadra"
summary

In [ ]:
# Límites de precio aprendidos SOLO con el entrenamiento: [0,1 x, 20 x] la mediana de cada ciudad, en USD
exclusions[exclusions["step"] == "price_bounds"].set_index("city")[["low_usd", "high_usd"]].round(1)

## 3. Guardar los conjuntos y el registro

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_csv(PROCESSED_DIR / "train.csv.gz", index=False, compression="gzip")
test.to_csv(PROCESSED_DIR / "test.csv.gz", index=False, compression="gzip")
exclusions.to_csv(REPORTS_DIR / "03_exclusiones.csv", index=False)
for path in (PROCESSED_DIR / "train.csv.gz", PROCESSED_DIR / "test.csv.gz", REPORTS_DIR / "03_exclusiones.csv"):
    print(f"{path.relative_to(PROCESSED_DIR.parents[1])}: {path.stat().st_size / 1e6:.1f} MB")

## 4. Línea base (solo con el entrenamiento)

Mediana de `price_usd` por ciudad × `room_type`, evaluada con 5 pliegues por anfitrión dentro del entrenamiento. Es el MAE
que el modelo de la etapa 4 tiene que bajar.

In [ ]:
baseline = baseline_cv_mae(train, n_splits=5)
print("MAE por pliegue (USD):", [round(m, 2) for m in baseline["per_fold"]])
print(f"MAE de la línea base: {baseline['mean']:.2f} ± {baseline['std']:.2f} USD por noche")

## 5. Qué copiar al informe

- El MAE de la línea base → `reports/01_eda.md` ("Qué son los datos" → "Línea base").
- Las filas quitadas por regla → `especificaciones/03-preprocesamiento.md`, sección 3.
- Comprobar las pruebas desde la raíz del repo: `pytest`.